# Breast Cancer Classification – Neural Network vs BDT

In this tutorial we compare two machine learning models:

- A small Neural Network (NN)
- A Boosted Decision Tree (BDT)

We use the Breast Cancer Wisconsin dataset to classify tumors as:

- 0 → Benign
- 1 → Malignant

Since this is a medical application, we will carefully evaluate:

- Training behavior (over/underfitting)
- Prediction distributions
- ROC curves
- Feature importance
- Sensitivity and specificity

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nairods/Scies4Free-FastML-tutorial/blob/main/part0.ipynb)

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_curve, roc_auc_score,
    confusion_matrix, classification_report
)
from sklearn.ensemble import GradientBoostingClassifier

import shap
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

%matplotlib inline
np.random.seed(42)

## 2. Load and Inspect the Dataset

In [ ]:
data = fetch_openml("heart-disease", version=1)

X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

print("Feature names:\n")
print(data.feature_names)

print("\nFirst 5 rows of data:")
display(X.head())

print("\nClass distribution:")
print(pd.Series(y).value_counts())

## 3. Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print("Training samples:", X_train.shape[0])
print("Test samples:", X_test.shape[0])

## 4. Scale Features (for Neural Network)

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

## 5. Build and Train a Small Neural Network

We use a sigmoid output layer for binary classification and optimize using binary crossentropy.

In [ ]:
nn = Sequential([
    Dense(16, activation='relu', input_shape=(X_train.shape[1],)),
    Dense(8, activation='relu'),
    Dense(1, activation='sigmoid')
])

nn.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history = nn.fit(
    X_train_scaled, y_train,
    validation_split=0.2,
    epochs=40,
    batch_size=16,
    verbose=0
)

## 6. Training History (Over/Underfitting)

In [ ]:
plt.figure()
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.title('Training vs Validation Loss')
plt.show()

## 7. Train a Boosted Decision Tree (BDT)

In [ ]:
bdt = GradientBoostingClassifier(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.1
)

bdt.fit(X_train, y_train)

## 8. Prediction Distributions

In [ ]:
nn_pred = nn.predict(X_test_scaled).ravel()
bdt_pred = bdt.predict_proba(X_test)[:,1]

plt.figure()
plt.hist(nn_pred[y_test==0], bins=30, alpha=0.6, label='Benign', color='blue')
plt.hist(nn_pred[y_test==1], bins=30, alpha=0.6, label='Malignant', color='orange')
plt.xlabel('Neural Network Output')
plt.ylabel('Events')
plt.legend()
plt.title('Prediction Distribution (NN)')
plt.show()

## 9. ROC Curve Comparison

In [ ]:
fpr_nn, tpr_nn, _ = roc_curve(y_test, nn_pred)
fpr_bdt, tpr_bdt, _ = roc_curve(y_test, bdt_pred)

plt.figure()
plt.plot(fpr_nn, tpr_nn, label=f'NN (AUC={roc_auc_score(y_test, nn_pred):.3f})')
plt.plot(fpr_bdt, tpr_bdt, label=f'BDT (AUC={roc_auc_score(y_test, bdt_pred):.3f})')
plt.plot([0,1],[0,1],'--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend()
plt.title('ROC Curve Comparison')
plt.show()

## 10. Feature Importance (BDT)

In [ ]:
importances = pd.Series(bdt.feature_importances_, index=X.columns)
importances.sort_values().plot(kind='barh')
plt.title('BDT Feature Importance')
plt.show()

## 11. Feature Importance (Neural Network via SHAP)

In [ ]:
explainer = shap.Explainer(nn, X_train_scaled)
shap_values = explainer(X_test_scaled[:200])
shap.summary_plot(shap_values, features=X_test.iloc[:200])

## 12. Medical Evaluation: Sensitivity vs Specificity

In medical diagnostics, missing a malignant tumor (false negative) is often worse than a false positive.
We therefore examine the confusion matrix and classification metrics.

In [ ]:
threshold = 0.5
y_pred_binary = (nn_pred > threshold).astype(int)

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_binary))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_binary))